# Неунижайка: файн-тюн Qwen3-8B на своей переписке

1. **Среда выполнения → Сменить среду выполнения → GPU T4** (или лучше, если есть).
2. Запусти ячейки по порядку (или «Выполнить всё»). На шаге 2 загрузи `train.jsonl` и `val.jsonl` из `data/finetune/`.
3. В конце в Google Drive появится папка `neunizhayka/` с GGUF-моделью и `Modelfile`. Ее нужно скачать в `models/` проекта.

LoRA-адаптер сохраняется в Drive сразу после обучения: если Colab отвалится на экспорте в GGUF, обучение не пропадет.

## 1. Установка

In [ ]:
%%capture
!pip install -q unsloth

## 2. Данные и Google Drive

In [ ]:
import os
from google.colab import drive, files

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/neunizhayka"
os.makedirs(OUT_DIR, exist_ok=True)

# Если файлы уже лежат в Drive (OUT_DIR), загрузка не нужна.
if not all(os.path.exists(f"{OUT_DIR}/{name}") for name in ("train.jsonl", "val.jsonl")):
    print("Выбери train.jsonl и val.jsonl из data/finetune/")
    for name, data in files.upload().items():
        with open(f"{OUT_DIR}/{name}", "wb") as f:
            f.write(data)

print(os.listdir(OUT_DIR))

## 3. Модель

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Форматирование

Диалоги переводятся в чат-шаблон Qwen3 без рассуждений (`enable_thinking=False`): так же модель будет работать в Ollama.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": f"{OUT_DIR}/train.jsonl",
    "validation": f"{OUT_DIR}/val.jsonl",
})


def to_text(batch):
    return {"text": [
        tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
        for messages in batch["messages"]
    ]}


dataset = dataset.map(to_text, batched=True, remove_columns=["messages"])
print(dataset)
print(dataset["train"][0]["text"][:1500])

## 5. Обучение

Лосс считается только по твоим ответам (`train_on_responses_only`): реплики собеседников модель видит как контекст, но не учится их писать.
Одна эпоха на T4 — примерно 30–60 минут. `eval_loss` должен падать вместе с `loss`; если он начал расти, а `loss` все еще падает — модель переобучается.

In [ ]:
import inspect

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# В разных версиях TRL параметры называются по-разному.
length_arg = "max_length" if "max_length" in inspect.signature(SFTConfig).parameters else "max_seq_length"
tokenizer_arg = "processing_class" if "processing_class" in inspect.signature(SFTTrainer).parameters else "tokenizer"

trainer = SFTTrainer(
    model=model,
    **{tokenizer_arg: tokenizer},
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        **{length_arg: MAX_SEQ_LENGTH},
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=20,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        output_dir="/content/outputs",
        seed=3407,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

## 6. Сохранить LoRA в Drive (страховка)

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
print("LoRA сохранена в", f"{OUT_DIR}/lora")

## 7. Быстрая проверка

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = open(f"{OUT_DIR}/train.jsonl", encoding="utf-8").readline()
SYSTEM = __import__("json").loads(SYSTEM)["messages"][0]["content"]

for question in [
    "здарова, че делаешь?",
    "го в субботу на шашлыки",
    "скинь 500 до пятницы плиз",
    "у меня кот заболел, сидим у ветеринара",
    "расскажи ка мне за ту самую",
    "ты вообще бот или живой?",
]:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.7, top_p=0.9, do_sample=True)
    answer = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"> {question}\n{answer}\n")

## 8. Экспорт в GGUF для Ollama

Занимает 10–20 минут (собирается llama.cpp, модель сливается с LoRA и квантуется в q4_k_m, ~5 ГБ).

In [ ]:
import glob
import shutil

model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_k_m")

gguf_files = [p for p in glob.glob("/content/gguf*/**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
print(gguf_files)
shutil.copy(gguf_files[0], f"{OUT_DIR}/neunizhayka-q4_k_m.gguf")
print("Готово:", os.listdir(OUT_DIR))

## 9. Дальше — на своем компьютере

1. Скачай `neunizhayka-q4_k_m.gguf` из Google Drive (`MyDrive/neunizhayka/`) в папку `models/` проекта.
2. В папке проекта:
   ```
   ollama create neunizhayka -f models/Modelfile
   ```
3. В `.env`:
   ```
   OLLAMA_MODEL=neunizhayka
   PERSONA_FILE=data/persona.finetune.txt
   FEW_SHOT_EXAMPLES=0
   ```